In [15]:
import random
from micrograd.engine import Value
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler

class Neuron:

    def __init__(self, nin):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1, 1))

    def __call__(self, x):
        act = sum((wi*xi for wi, xi in zip(self.w, x)), self.b)
        return act

class Layer:

    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs


class MLP:
    def __init__(self, nin, outs):
        sizes = [nin] + outs  # gives [4, 8, 3]
        self.layers = [Layer(sizes[i], sizes[i+1]) for i in range(len(sizes)-1)]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self): 
        return [p for layer in self.layers for neuron in layer.neurons for p in neuron.w + [neuron.b]]


# load data
iris = load_iris()
scaler = StandardScaler()
X = scaler.fit_transform(iris.data) 
y = iris.target 
 
model = MLP(4, [16, 8, 3])

for step in range(300):
    # zero grads
    for p in model.parameters():
        p.grad = 0.0
    
    total_loss = Value(0.0)
    
    for i, sample in enumerate(X):
        x_sample = [Value(v) for v in sample]
        output = model(x_sample)
        target = [Value(1.0 if j == y[i] else 0.0) for j in range(3)]
        loss = sum((o - t)**2 for o, t in zip(output, target))
        total_loss = total_loss + loss
    
    total_loss.backward()
    
    for p in model.parameters():
        p.data -= 0.0001 * p.grad
    
    if step % 10 == 0:
        print(f"Step {step}, Loss: {total_loss.data:.4f}")

Step 0, Loss: 6830.3882
Step 10, Loss: 73.3122
Step 20, Loss: 52.5667
Step 30, Loss: 47.3831
Step 40, Loss: 45.2048
Step 50, Loss: 44.0011
Step 60, Loss: 43.2196
Step 70, Loss: 42.6641
Step 80, Loss: 42.2483
Step 90, Loss: 41.9271
Step 100, Loss: 41.6738
Step 110, Loss: 41.4710
Step 120, Loss: 41.3066
Step 130, Loss: 41.1719
Step 140, Loss: 41.0607
Step 150, Loss: 40.9681
Step 160, Loss: 40.8905
Step 170, Loss: 40.8250
Step 180, Loss: 40.7695
Step 190, Loss: 40.7222
Step 200, Loss: 40.6816
Step 210, Loss: 40.6468
Step 220, Loss: 40.6168
Step 230, Loss: 40.5907
Step 240, Loss: 40.5682
Step 250, Loss: 40.5485
Step 260, Loss: 40.5314
Step 270, Loss: 40.5164
Step 280, Loss: 40.5033
Step 290, Loss: 40.4919


In [4]:
import numpy as np
predicted = np.argmax([o.data for o in output])
print(predicted)  # gives 0, 1, or 2
print(y[0])       # real answer


0
0


In [17]:
count = 0  # outside the loop

for i, sample in enumerate(X):
    x_sample = [Value(v) for v in sample]
    output = model(x_sample)
    predicted = np.argmax([o.data for o in output])
    if predicted == y[i]:  # compare to correct label
        count += 1

accuracy = count / len(X)
print(f"Accuracy: {accuracy * 100:.1f}%")

Accuracy: 85.3%
